# COGANT Demo Notebook

This notebook is the end-to-end demo for **COGANT** (Codebase-to-GNN Translation Engine).
It walks through the four user-facing surfaces of the engine on the bundled
`examples/control_positive/calculator` fixture:

1. The forward pipeline (`cogant.api.pipeline.PipelineRunner`)
2. Markov blanket extraction (`cogant.markov.MarkovBlanketExtractor`)
3. Reverse synthesis and round-trip scoring (`cogant.reverse.verify_repo_roundtrip`)
4. Per-node role justification (`cogant.cli.explain.explain_node`)

Every cell is idempotent and re-runnable from a clean kernel. The notebook does
not mutate the source tree and writes all reverse-synthesis output to a
temporary directory that is cleaned up at the end.

> COGANT's GNN is the *Active Inference Institute's* **Generalized Notation
> Notation** — a structured state-space / process-model notation. It is
> **not** graph neural networks.

## Cell 1 - Install, import, and show version

COGANT is installed from the repository root via `uv sync`. The notebook
expects to be launched from inside the repository so that the editable
install is already on `sys.path`. We import the primary entry points and
print the package version so the reader knows which build they are running
against.

In [ ]:
# Cell 1: imports + version check.
#
# If COGANT is not importable, uncomment the install line below. When run from
# the repo root the editable install put in place by `uv sync` is already on
# sys.path, so no network access is required.
# !uv pip install -e .

import json
import shutil
import tempfile
from collections import Counter
from pathlib import Path

import cogant
from cogant.api.bundle import ArtifactKey
from cogant.api.pipeline import PipelineConfig, PipelineRunner
from cogant.cli.explain import explain_node
from cogant.markov import MarkovBlanketExtractor
from cogant.reverse import (
    verify_repo_roundtrip,
)

print(f"cogant version: {cogant.__version__}")
print(f"rust backend available: {cogant._RUST_AVAILABLE}")
if cogant.__rust_version__:
    print(f"rust backend version: {cogant.__rust_version__}")

## Cell 2 - Analyze the calculator example

We run the static forward pipeline (`ingest -> static -> normalize -> graph
-> translate -> statespace`) on the bundled calculator fixture. `skip_dynamic`
keeps the run hermetic - no coverage database is required.

The resulting `Bundle` exposes the program graph, semantic mappings, and
state-space model via `get_artifact(ArtifactKey.*)`. We print node/edge
counts and the distribution of AI role assignments produced by the
translation engine.

In [ ]:
# Cell 2: run the pipeline on the calculator fixture.
CALCULATOR_PATH = Path("examples/control_positive/calculator").resolve()
assert CALCULATOR_PATH.exists(), f"calculator fixture missing: {CALCULATOR_PATH}"

runner = PipelineRunner()
config = PipelineConfig(
    stages=["ingest", "static", "normalize", "graph", "translate", "statespace"],
    skip_dynamic=True,
)
bundle = runner.run(str(CALCULATOR_PATH), config)

program_graph = bundle.get_artifact(ArtifactKey.PROGRAM_GRAPH, required=True)
semantic_mappings = bundle.get_artifact(ArtifactKey.SEMANTIC_MAPPINGS) or {}

role_counts = Counter()
for mapping in semantic_mappings.values():
    kind = getattr(mapping.kind, "value", str(mapping.kind))
    role_counts[kind] += 1

print(f"target:            {bundle.target}")
print(f"errors:            {len(bundle.errors)}")
print(f"graph nodes:       {len(program_graph.nodes)}")
print(f"graph edges:       {len(program_graph.edges)}")
print(f"semantic mappings: {len(semantic_mappings)}")
print("role assignments:")
for role, count in sorted(role_counts.items(), key=lambda kv: (-kv[1], kv[0])):
    print(f"  {role:<15} {count}")

## Cell 3 - Markov blanket partition

The `MarkovBlanketExtractor` slices the program graph into an Active
Inference four-way partition:

| Role       | Symbol | Meaning                                           |
|-----------|:------:|---------------------------------------------------|
| internal  |  mu    | Inside the system of interest, no external edges  |
| sensory   |  s     | Boundary node with incoming edges from outside    |
| active    |  a     | Boundary node with outgoing edges to outside      |
| external  |  eta   | Everything else                                   |

The `auto` strategy picks the most cohesive module as the system of
interest. For the single-file calculator fixture this usually gives a
large `internal` set with a thin `sensory`/`active` boundary.

In [ ]:
# Cell 3: Markov blanket partition.
extractor = MarkovBlanketExtractor(program_graph)
blanket = extractor.extract(strategy="auto")

partition = {
    "internal": len(blanket.internal_ids),
    "sensory": len(blanket.sensory_ids),
    "active": len(blanket.active_ids),
    "external": len(blanket.external_ids),
}
total = sum(partition.values()) or 1

print("Markov blanket partition:")
for role, count in partition.items():
    pct = 100.0 * count / total
    bar = "#" * int(pct / 5)
    print(f"  {role:<9} {count:>3}  {pct:5.1f}%  {bar}")

print(f"\nboundary (s + a):    {len(blanket.boundary_ids)}")
print(f"auto strategy reason: {blanket.metadata.get('auto_reason', 'n/a')}")
print(f"chosen class id:      {blanket.metadata.get('chosen_class_id', 'n/a')}")

## Cell 4 - Reverse: synthesize a package from the GNN

`cogant.reverse` runs the pipeline in the other direction: given the GNN
markdown produced by the forward pass, it plans and emits a Python package
whose state, observations, and actions mirror the model topology.

`verify_repo_roundtrip` does the full forward -> reverse -> forward loop:

1. Runs forward on the calculator -> emits `model.gnn.md`.
2. Parses the GNN back into a `ReverseGNNModel`.
3. Synthesizes a Python package from the plan.
4. Runs forward again on the synthesized package.
5. Compares the two runs' role multisets and state-space shapes.

We keep the intermediate artifacts on disk so later cells can poke at the
synthesized package.

In [ ]:
# Cell 4: reverse synthesis round-trip.
ROUNDTRIP_DIR = Path(tempfile.mkdtemp(prefix="cogant-demo-roundtrip-"))
print(f"working dir: {ROUNDTRIP_DIR}")

roundtrip = verify_repo_roundtrip(
    str(CALCULATOR_PATH),
    output_dir=ROUNDTRIP_DIR,
)

print(f"isomorphic:    {roundtrip.structurally_isomorphic}")
print(f"role_match:    {roundtrip.role_preservation_score:.2%}")
print(f"original:      {roundtrip.original_roles}")
print(f"synthesized:   {roundtrip.synthesized_roles}")
print(f"shape_match:   {roundtrip.shape_match}")

gnn_path = ROUNDTRIP_DIR / "forward" / "model.gnn.md"
if gnn_path.exists():
    print(f"\nGNN markdown: {gnn_path} ({gnn_path.stat().st_size} bytes)")

if roundtrip.package_path and roundtrip.package_path.exists():
    synth_files = sorted(p.name for p in roundtrip.package_path.iterdir())
    print(f"synthesized package at: {roundtrip.package_path}")
    print(f"files: {synth_files}")

## Cell 5 - Round-trip role match score

The round-trip score is the fraction of role labels in the *original*
multiset that are recovered in the *synthesized* multiset. A score of
`1.0` means every role from the source GNN reappears after synthesize +
re-forward. The default threshold for weak isomorphism is `0.5`.

In [ ]:
# Cell 5: round-trip score breakdown.
orig = Counter(roundtrip.original_roles)
synth = Counter(roundtrip.synthesized_roles)

all_roles = sorted(set(orig) | set(synth))
print(f"{'role':<15} {'original':>8} {'synth':>8} {'matched':>8}")
print("-" * 42)
for role in all_roles:
    o = orig.get(role, 0)
    s = synth.get(role, 0)
    m = min(o, s)
    print(f"{role:<15} {o:>8} {s:>8} {m:>8}")

overlap = sum((orig & synth).values())
total_orig = sum(orig.values()) or 1
score = overlap / total_orig
print("-" * 42)
print(f"overlap: {overlap} / {total_orig}")
print(f"role_preservation_score: {score:.4f}")
print(f"structurally_isomorphic (>= 0.5): {score >= 0.5}")

## Cell 6 - `cogant explain` on a single node

`explain_node(repo_path, node_query)` re-runs the static pipeline and asks
every translation rule to justify its decision on the requested node. The
result records the assigned AI role, which rules fired vs considered, and
the node's Markov blanket role.

For the calculator we explain `input_digit`, which should be classified
as an **action** (it mutates hidden state in response to a user event).

In [ ]:
# Cell 6: explain a single node.
TARGET_NODE = "input_digit"
explanation = explain_node(str(CALCULATOR_PATH), TARGET_NODE)

print(f"node:          {explanation.node_name} ({explanation.node_kind})")
print(f"assigned role: {explanation.assigned_role}")
print(f"blanket role:  {explanation.blanket_role}")
print(f"mapping label: {explanation.mapping_label}")

print(f"\nrules fired ({len(explanation.rules_fired)}):")
for rx in explanation.rules_fired[:5]:
    print(f"  [p={rx.priority:>3}] {rx.rule_name}: {rx.reason}")
    for line in rx.evidence[:3]:
        print(f"      - {line}")

print(f"\nrules considered: {len(explanation.rules_considered)}")

## Cell 7 - Summary table

A single pane of glass with every metric we computed above. This is the
cell a reviewer should screenshot.

In [ ]:
# Cell 7: assemble summary metrics.
summary = {
    "cogant_version": cogant.__version__,
    "target": str(CALCULATOR_PATH),
    "pipeline_errors": len(bundle.errors),
    "graph_nodes": len(program_graph.nodes),
    "graph_edges": len(program_graph.edges),
    "semantic_mappings": len(semantic_mappings),
    "roles": dict(role_counts),
    "blanket_internal": len(blanket.internal_ids),
    "blanket_sensory": len(blanket.sensory_ids),
    "blanket_active": len(blanket.active_ids),
    "blanket_external": len(blanket.external_ids),
    "roundtrip_structurally_isomorphic": roundtrip.structurally_isomorphic,
    "roundtrip_role_match": round(roundtrip.role_preservation_score, 4),
    "roundtrip_shape_match": roundtrip.shape_match,
    "explained_node": explanation.node_name,
    "explained_role": explanation.assigned_role,
    "explained_blanket_role": explanation.blanket_role,
    "rules_fired": len(explanation.rules_fired),
    "rules_considered": len(explanation.rules_considered),
}

print("COGANT demo summary")
print("=" * 60)
key_w = max(len(k) for k in summary)
for key, value in summary.items():
    print(f"  {key:<{key_w}}  {value}")
print("=" * 60)

# JSON copy for machine consumers.
print("\nJSON:")
print(json.dumps(summary, indent=2, default=str))

# Clean up the temporary round-trip directory.
shutil.rmtree(ROUNDTRIP_DIR, ignore_errors=True)
print(f"\nCleaned up {ROUNDTRIP_DIR}")